In [1]:
import os
import asyncio
import random
from prisma import Prisma
from datetime import datetime
from typing import Optional

# Import your old in-memory data
from db import get_all_calls  # Replace with actual path or inline data

def map_call_type(old_type: str) -> Optional[str]:
    old_type = old_type.lower()
    if old_type == 'fire':
        return 'Fire'
    elif old_type == 'hospital':
        return 'Medical'
    elif old_type == 'police':
        return 'Police'
    return None

def map_severity(old_severity: str) -> Optional[str]:
    old_severity = old_severity.upper()
    if old_severity in ['MODERATE', 'MEDIUM']:
        return 'Medium'
    elif old_severity == 'LOW':
        return 'Low'
    elif old_severity == 'HIGH':
        return 'High'
    elif old_severity == 'CRITICAL':
        return 'Critical'
    return None

def choose_sentiment(emotions: list) -> Optional[str]:
    # Emotions: [{"emotion": "fear", "intensity": 0.8}, ...]
    if not emotions:
        return None
    max_emo = max(emotions, key=lambda e: e.get('intensity',0))
    return max_emo.get('emotion')

def guess_lat_lon(location_name: str) -> (float, float):
    # Guess some coordinates
    lat = random.uniform(37.0, 38.0)     # around Northern California
    lon = random.uniform(-122.0, -121.0)
    return lat, lon


In [4]:
db = Prisma()
await db.connect()
    
calls_data = get_all_calls().values()
print(calls_data)

dict_values([{'id': '1', 'time': '2024-12-20T16:00:02.402524', 'transcript': [{'role': 'assistant', 'content': "9-1-1, what's your emergency?"}, {'role': 'user', 'content': "There's a fire in my apartment building!"}, {'role': 'assistant', 'content': "I understand there's a fire. What's your location_name?"}, {'role': 'user', 'content': '123 Main Street, Apartment 4B'}], 'emotions': [{'emotion': 'fear', 'intensity': 0.8}, {'emotion': 'stress', 'intensity': 0.9}, {'emotion': 'urgency', 'intensity': 0.95}], 'phone': '555-123-4567', 'recommendation': 'Evacuate immediately. Do not use elevators.', 'severity': 'MODERATE', 'type': 'fire', 'name': 'John Doe', 'title': 'Apartment Fire on Main Street', 'summary': 'Caller reported a fire in their apartment building. Immediate evacuation required.', 'location_name': '123 Main Street, Apartment 4B'}, {'id': '2', 'time': '2024-12-20T14:34:02.402538', 'transcript': [{'role': 'assistant', 'content': "9-1-1, what's your emergency?"}, {'role': 'user', 

In [5]:
for call in calls_data:
    phone = call.get('phone')
    # If no phone number, skip the call
    if not phone:
        continue
    
    # Check if user with this phone number exists
    user = await db.user.find_first(where={'phoneNumber': phone})
    if user is None:
        # No registered user with this phone, skip
        continue
    
    old_type = call.get('type', '')
    call_type = map_call_type(old_type)
    
    old_severity = call.get('severity', '')
    severity = map_severity(old_severity)
    
    summary = call.get('summary')
    location_name = call.get('location_name')
    name = call.get('name')
    emotions = call.get('emotions', [])
    sentiment = choose_sentiment(emotions)
    
    recommendation = call.get('recommendation')
    
    # Guess lat/lon
    lat, lon = guess_lat_lon(location_name or "")

    # Parse the old 'time' for createdAt if possible
    time_str = call.get('time')
    try:
        created_at = datetime.fromisoformat(time_str)
    except:
        created_at = datetime.now()
    
    # Create Call record linked to the existing user
    new_call = await db.call.create(data={
        'status': 'Active',
        'inProgress': True,
        'createdAt': created_at,
        'userId': user.id
    })

    # Create CallAnalytics
    await db.callanalytics.create(data={
        'callId': new_call.id,
        'type': call_type,
        'severity': severity,
        'summary': summary,
        'sentiment': sentiment,
        'topics': [],
        'location': location_name,
        'latitude': lat,
        'longitude': lon,
        'name': name,
        'address': location_name,
        'object': recommendation
    })

    # Create Messages
    transcript = call.get('transcript', [])
    for msg in transcript:
        role = msg.get('role', 'assistant')
        speaker = 'Assistant' if role == 'assistant' else 'Caller'
        content = msg.get('content', '')
        await db.message.create(data={
            'callId': new_call.id,
            'speaker': speaker,
            'content': content,
        })

await db.disconnect()

DataError: The column `User.credits` does not exist in the current database.